# 07 - Hybrid Fraud Scoring & Multi-Layer Decision Engine

This notebook implements and demonstrates the weighted hybrid risk score combining Machine Learning (0.5), Behavioral Analytics (0.3), and Deterministic Rules (0.2).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from src.models.hybrid_scorer import HybridFraudScorer
from src.rules.fraud_rules import FraudRuleEngine

hybrid = HybridFraudScorer()
print(f'Hybrid Weights: ML={hybrid.w_ml}, Behavior={hybrid.w_behavior}, Rules={hybrid.w_rule}')
print(f'Thresholds: Suspicious >= {hybrid.suspicious_threshold}, Fraud >= {hybrid.fraud_threshold}')

Hybrid Weights: ML=0.5, Behavior=0.3, Rules=0.2
Thresholds: Suspicious >= 0.4, Fraud >= 0.7


In [2]:
# Case 1: Legitimate Transaction
c1 = hybrid.calculate_score(ml_score=0.02, behavior_score=0.05, rule_score=0.0)
print('Case 1 (Legitimate):', c1)

# Case 2: Suspicious Rapid Burst
c2 = hybrid.calculate_score(
    ml_score=0.55,
    behavior_score=0.45,
    rule_score=0.35,
    rule_explanations=['Rapid velocity: 6 transactions in 2 minutes.'],
    behavior_explanations=['High velocity: 7 txns in 5 minutes.']
)
print('\nCase 2 (Suspicious):', c2)

# Case 3: Confirmed High-Value Takeover Fraud
c3 = hybrid.calculate_score(
    ml_score=0.92,
    behavior_score=0.85,
    rule_score=0.70,
    rule_explanations=['Impossible travel from New York to Singapore in 15 mins.', 'Amount spike 12x normal average.'],
    behavior_explanations=['Unusual location and timing outside normal hours.']
)
print('\nCase 3 (Fraud):', c3)

Case 1 (Legitimate): {'ml_score': 0.02, 'behavior_score': 0.05, 'rule_score': 0.0, 'final_fraud_score': 0.025, 'fraud_decision': 'LEGITIMATE', 'explanation': 'Transaction patterns consistent with normal customer spending.', 'reasons_list': []}

Case 2 (Suspicious): {'ml_score': 0.55, 'behavior_score': 0.45, 'rule_score': 0.35, 'final_fraud_score': 0.48, 'fraud_decision': 'SUSPICIOUS', 'explanation': 'Rapid velocity: 6 transactions in 2 minutes. | High velocity: 7 txns in 5 minutes. | ML Classifier high fraud probability: 55.0%.', 'reasons_list': ['Rapid velocity: 6 transactions in 2 minutes.', 'High velocity: 7 txns in 5 minutes.', 'ML Classifier high fraud probability: 55.0%.']}

Case 3 (Fraud): {'ml_score': 0.92, 'behavior_score': 0.85, 'rule_score': 0.7, 'final_fraud_score': 0.855, 'fraud_decision': 'FRAUD', 'explanation': 'Impossible travel from New York to Singapore in 15 mins. | Amount spike 12x normal average. | Unusual location and timing outside normal hours. | ML Classifier h

## Notebook Explanation

### What was done
- Implemented the strictly required hybrid scoring formula:
  `Fraud Score = 0.5 * ML Score + 0.3 * Behavior Score + 0.2 * Rule Score`.
- Evaluated the 7 deterministic fraud rules.
- Implemented decision boundaries: `LEGITIMATE` (< 0.40), `SUSPICIOUS` (0.40 - 0.69), `FRAUD` (>= 0.70).
- Generated aggregated human-readable explanations for flagged alerts.

### Why it was done
A hybrid architecture combines the generalization of statistical machine learning, the personalized context of customer behavior, and the absolute safety of deterministic security rules.

### Input
- ML probability score, behavioral anomaly score, and triggered rule scores.

### Output
- Final composite risk score in [0.0, 1.0].
- Alert classification (`LEGITIMATE`, `SUSPICIOUS`, or `FRAUD`).
- Consolidated human-readable explanations.

### Connection to next stage
The hybrid scorer powers the end-to-end streaming validation in `08_streaming_pipeline_validation.ipynb` and populates the PostgreSQL database for the Streamlit dashboard.

### Implementation decisions
- Weights are strictly fixed at 0.5 (ML), 0.3 (Behavior), and 0.2 (Rules) as mandated by the specification.
- Explainable alert text is preserved alongside numerical scores for auditor traceability.